In [1]:
import os
from pathlib import Path
import pickle
import requests
import mercantile
import mapbox_vector_tile
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from dotenv import load_dotenv
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# ── Load token ──
load_dotenv()
MLY_ACCESS_TOKEN = os.getenv("MLY_ACCESS_TOKEN")

if not MLY_ACCESS_TOKEN:
    raise ValueError("MLY_ACCESS_TOKEN not found in .env")

# ── Paths ──
PROJECT_ROOT = Path(".").resolve()
SEG_DIR = PROJECT_ROOT / "segmentation"
SEG_DIR.mkdir(exist_ok=True)

print(f"Project: {PROJECT_ROOT}")
print(f"Token: {MLY_ACCESS_TOKEN[:15]}...")

Project: D:\DOWNLOADS\nasim\jupyters\maps\map\new code for git\Project_Mapilary_GIT
Token: MLY|38779264065...


In [2]:
def tile_to_lonlat(x, y, tile, zoom, extent=4096):
    b = mercantile.xy_bounds(tile.x, tile.y, zoom)
    fx, fy = x / extent, y / extent
    x_m = b.left + fx * (b.right - b.left)
    y_m = b.top - fy * (b.top - b.bottom)
    return mercantile.lnglat(x_m, y_m)

In [3]:


# ── چک کن اگه قبلاً ذخیره شده، لود کن ──
CACHE_FILE = PROJECT_ROOT / "data" / "sequences.pkl"

if CACHE_FILE.exists():
    with open(CACHE_FILE, "rb") as f:
        sequences = pickle.load(f)
    print(f"✅ Loaded from cache: {len(sequences)} sequences")
else:
    # ── Fetch from API ──
    CENTER_LON, CENTER_LAT = 4.89261, 52.37312
    ZOOM = 14
    
    tile = mercantile.tile(lng=CENTER_LON, lat=CENTER_LAT, zoom=ZOOM)
    url = f"https://tiles.mapillary.com/maps/vtp/mly1_public/2/{ZOOM}/{tile.x}/{tile.y}"
    resp = requests.get(url, params={"access_token": MLY_ACCESS_TOKEN})
    resp.raise_for_status()
    decoded = mapbox_vector_tile.decode(resp.content)
    
    sequences = []
    for feat in decoded["sequence"]["features"]:
        seq_id = feat["properties"]["id"]
        geom = feat["geometry"]
        lines = geom["coordinates"] if geom["type"] == "MultiLineString" else [geom["coordinates"]]
        lonlat_lines = [
            [tile_to_lonlat(x, y, tile, ZOOM) for x, y in line]
            for line in lines
        ]
        sequences.append({
            "sequence_id": seq_id,
            "lines": lonlat_lines,
            "quality_score": feat["properties"].get("quality_score"),
            "is_pano": feat["properties"].get("is_pano", False),
            "foot": feat["properties"].get("foot", False),
        })
    
    # ── ذخیره ──
    CACHE_FILE.parent.mkdir(exist_ok=True)
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(sequences, f)
    print(f"✅ Fetched and cached: {len(sequences)} sequences")

✅ Fetched and cached: 615 sequences


In [4]:
def extract_images_from_sequence(seq_id, max_imgs=None):
    url = "https://graph.mapillary.com/image_ids"
    params = {"access_token": MLY_ACCESS_TOKEN, "sequence_id": seq_id}
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        return []
    
    image_ids = [obj["id"] for obj in response.json().get("data", [])]
    if max_imgs:
        image_ids = image_ids[:max_imgs]
    
    images = []
    for img_id in image_ids:
        url = f"https://graph.mapillary.com/{img_id}"
        params = {
            "access_token": MLY_ACCESS_TOKEN,
            "fields": "id,computed_geometry,geometry,captured_at,compass_angle,is_pano,thumb_2048_url",
        }
        r = requests.get(url, params=params)
        if r.status_code != 200:
            continue
        
        data = r.json()
        geom = data.get("computed_geometry") or data.get("geometry")
        if not geom:
            continue
        
        lon, lat = geom["coordinates"]
        images.append({
            "seq_id": seq_id,
            "id": img_id,
            "lat": lat,
            "lon": lon,
            "captured_at": data.get("captured_at"),
            "compass_angle": data.get("compass_angle"),
            "is_pano": data.get("is_pano", False),
            "url": data.get("thumb_2048_url"),
        })
    
    images.sort(key=lambda x: x["captured_at"] or 0)
    return images

In [5]:
MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_NAME)
model.eval()

id2label = model.config.id2label

ADE20K_MAP = {
    "sky":        (["sky"], (135, 206, 235)),
    "building":   (["building", "house", "skyscraper", "wall", "fence"], (169, 169, 169)),
    "vegetation": (["tree", "plant", "grass", "flower", "earth", "mountain"], (34, 139, 34)),
    "road":       (["road", "path", "runway"], (105, 105, 105)),
    "sidewalk":   (["sidewalk", "pavement"], (222, 184, 135)),
    "vehicle":    (["car", "bus", "truck", "van", "motorcycle", "bicycle"], (220, 20, 60)),
    "person":     (["person"], (255, 215, 0)),
}

print("✅ Model loaded")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

✅ Model loaded


In [6]:
def segment_and_save(img_id, url):
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            return None
        
        img = Image.open(BytesIO(r.content)).convert("RGB")
        orig = np.array(img)
        
        inputs = processor(images=img, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            upsampled = torch.nn.functional.interpolate(
                logits, size=img.size[::-1], mode="bilinear", align_corners=False,
            )
            pred = upsampled.argmax(1).squeeze().cpu().numpy()
        
        h, w = pred.shape
        mask_rgb = np.zeros((h, w, 3), dtype=np.uint8)
        result = {}
        total = pred.size
        
        for group, (names, color) in ADE20K_MAP.items():
            group_mask = np.zeros_like(pred, dtype=bool)
            count = 0
            for lid, lname in id2label.items():
                if any(n in lname.lower() for n in names):
                    m = (pred == lid)
                    group_mask |= m
                    count += int(m.sum())
            result[group] = round(count / total * 100, 2)
            mask_rgb[group_mask] = color
        
        overlay = (orig * 0.5 + mask_rgb * 0.5).astype(np.uint8)
        
        mask_path = SEG_DIR / f"{img_id}_mask.png"
        overlay_path = SEG_DIR / f"{img_id}_overlay.jpg"
        Image.fromarray(mask_rgb).save(mask_path)
        Image.fromarray(overlay).save(overlay_path, quality=85)
        
        return {
            "segmentation": result,
            "mask_path": str(mask_path),
            "overlay_path": str(overlay_path),
        }
    except Exception as e:
        print(f"❌ {img_id}: {e}")
        return None

In [12]:
NUM_SEQUENCES = 2
MAX_PER_SEQUENCE = 30

meta = []
for i, seq in enumerate(sequences[:NUM_SEQUENCES]):
    seq_id = seq["sequence_id"]
    print(f"[{i+1}/{NUM_SEQUENCES}] {seq_id[:12]}...")
    images = extract_images_from_sequence(seq_id, max_imgs=MAX_PER_SEQUENCE)
    meta.extend(images)
    print(f"   → {len(images)} images")

sequences_list = list(dict.fromkeys(d["seq_id"] for d in meta))

print(f"\n✅ {len(sequences_list)} sequences")
print(f"✅ {len(meta)} images")

for i, img in enumerate(meta):
    print(f"[{i+1}/{len(meta)}] {img['id'][:12]}...")
    result = segment_and_save(img["id"], img["url"])
    if result:
        img["segmentation"] = result["segmentation"]
        img["mask_path"] = result["mask_path"]
        img["overlay_path"] = result["overlay_path"]
        print(f"   ✅ sky={result['segmentation']['sky']}%, building={result['segmentation']['building']}%")
    else:
        img["segmentation"] = None
        print(f"   ❌ failed")

[1/2] T79Mb0cVJdKA...
   → 30 images
[2/2] EXfdPov1FJBu...
   → 30 images

✅ 2 sequences
✅ 60 images
[1/60] 956082797212...
   ✅ sky=5.61%, building=47.71%
[2/60] 133168097548...
   ✅ sky=6.02%, building=42.94%
[3/60] 254934531216...
   ✅ sky=8.28%, building=45.01%
[4/60] 284692334898...
   ✅ sky=9.88%, building=41.24%
[5/60] 141322903416...
   ✅ sky=7.5%, building=22.74%
[6/60] 978583478195...
   ✅ sky=7.65%, building=1.36%
[7/60] 250735974969...
   ✅ sky=1.55%, building=11.07%
[8/60] 961109186672...
   ✅ sky=5.58%, building=26.87%
[9/60] 168747934259...
   ✅ sky=1.94%, building=11.78%
[10/60] 160210220192...
   ✅ sky=5.91%, building=14.48%
[11/60] 146107874841...
   ✅ sky=3.73%, building=15.4%
[12/60] 162359864871...
   ✅ sky=8.01%, building=7.53%
[13/60] 160217684420...
   ✅ sky=1.57%, building=17.2%
[14/60] 964908683134...
   ✅ sky=10.99%, building=26.04%
[15/60] 839545095318...
   ✅ sky=15.24%, building=39.32%
[16/60] 154463345041...
   ✅ sky=9.51%, building=33.95%
[17/60] 4535413

In [13]:
# ═══════════════════════════════════════════
# Cell: Build HTML Viewer
# ═══════════════════════════════════════════

import json

# ── Add overlay URLs (relative for HTML) ──
for img in meta:
    if img.get("overlay_path"):
        img["overlay_url"] = img["overlay_path"].replace("\\", "/")

# ── HTML Template ──
html = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        html, body { height: 100%; font-family: sans-serif; overflow: hidden; }
        #container { display: flex; height: 100vh; }
        #map { flex: 1; height: 100%; position: relative; }
        #seq-selector {
            position: absolute; top: 10px; left: 50px; z-index: 1000;
            background: white; padding: 8px 12px; border-radius: 4px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3); font-size: 13px;
        }
        #seq-selector select {
            padding: 4px 8px; border: 1px solid #ccc; border-radius: 4px;
            font-size: 13px; cursor: pointer; min-width: 200px;
        }
        #viewer {
            flex: 1; display: flex; flex-direction: column;
            background: #1a1a1a; border-left: 2px solid #333;
        }
        #photo {
            flex: 1; display: flex; align-items: center;
            justify-content: center; overflow: hidden; padding: 10px;
            position: relative; min-height: 0;
        }
        #photo img {
            max-width: 100%; max-height: 100%; object-fit: contain;
            border-radius: 4px; box-shadow: 0 4px 20px rgba(0,0,0,0.5);
        }
        #photo-tabs {
            position: absolute; top: 20px; left: 20px;
            display: flex; gap: 6px; z-index: 10;
        }
        #photo-tabs button {
            padding: 4px 12px; font-size: 12px;
            background: rgba(0,0,0,0.7); color: white;
            border: 1px solid #555; border-radius: 4px; cursor: pointer;
        }
        #photo-tabs button.active { background: #4CAF50; }
        #seg-stats {
            padding: 8px; background: #222; color: white;
            border-top: 1px solid #333;
            display: flex; flex-wrap: wrap; gap: 6px;
            font-size: 11px;
        }
        .seg-item {
            display: flex; align-items: center; gap: 4px;
            background: #2a2a2a; padding: 3px 8px; border-radius: 4px;
        }
        .seg-color { width: 10px; height: 10px; border-radius: 2px; }
        .seg-label { color: #aaa; }
        .seg-value { color: #fff; font-weight: bold; }
        #chart-container {
            height: 180px; background: #1a1a1a;
            border-top: 1px solid #333; flex-shrink: 0;
        }
        #controls {
            padding: 10px; background: #222; color: white;
            display: flex; gap: 10px; align-items: center;
            border-top: 1px solid #333; flex-wrap: wrap;
        }
        button {
            padding: 8px 16px; cursor: pointer; background: #4CAF50;
            color: white; border: none; border-radius: 4px; font-size: 14px;
        }
        button:hover:not(:disabled) { background: #45a049; }
        button:disabled { background: #444; cursor: not-allowed; }
        #counter { font-weight: bold; font-size: 14px; padding: 0 10px; }
        #info {
            margin-left: auto; font-size: 12px; color: #aaa;
            text-align: right; line-height: 1.4;
        }
        #info b { color: #fff; }
    </style>
</head>
<body>
    <div id="container">
        <div id="map">
            <div id="seq-selector">
                <label>Sequence: </label>
                <select id="seq-dropdown"></select>
            </div>
        </div>
        <div id="viewer">
            <div id="photo">
                <div id="photo-tabs">
                    <button id="tab-orig" class="active">Original</button>
                    <button id="tab-overlay">Segmentation</button>
                </div>
                <img id="current-photo" src="" alt="Loading...">
            </div>
            <div id="seg-stats"></div>
            <div id="chart-container"></div>
            <div id="controls">
                <button id="prev">◀ Prev</button>
                <button id="next">Next ▶</button>
                <span id="counter">0 / 0</span>
                <span id="info"></span>
            </div>
        </div>
    </div>
    <script>
        const META = META_JSON;
        const SEQUENCES = SEQ_JSON;
        
        const PALETTE = [
            '#e6194B', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
            '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
        ];
        
        const SEG_COLORS = {
            sky: '#87CEEB', building: '#A9A9A9', vegetation: '#228B22',
            road: '#696969', sidewalk: '#DEB887', vehicle: '#DC143C',
            person: '#FFD700',
        };
        
        const bySeq = {};
        META.forEach(d => {
            if (!bySeq[d.seq_id]) bySeq[d.seq_id] = [];
            bySeq[d.seq_id].push(d);
        });
        
        Object.keys(bySeq).forEach(seqId => {
            bySeq[seqId].sort((a, b) => (a.captured_at || 0) - (b.captured_at || 0));
        });
        
        const map = L.map('map').setView([META[0].lat, META[0].lon], 16);
        L.tileLayer('https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            { attribution: 'Google Satellite', maxZoom: 21 }).addTo(map);
        
        const seqLayers = {};
        const seqColors = {};
        
        SEQUENCES.forEach((seqId, i) => {
            const imgs = bySeq[seqId] || [];
            const color = PALETTE[i % PALETTE.length];
            seqColors[seqId] = color;
            
            const group = L.layerGroup().addTo(map);
            
            if (imgs.length > 1) {
                L.polyline(imgs.map(d => [d.lat, d.lon]),
                    { color: color, weight: 3, opacity: 0.6 }).addTo(group);
            }
            
            const pointMarkers = [];
            imgs.forEach((d, j) => {
                const marker = L.circleMarker([d.lat, d.lon], {
                    radius: 5, color: color, fillColor: color,
                    fillOpacity: 0.8, weight: 2,
                }).addTo(group);
                
                marker.on('click', () => {
                    document.getElementById('seq-dropdown').value = seqId;
                    switchSequence(seqId);
                    goTo(j);
                });
                pointMarkers.push(marker);
            });
            
            seqLayers[seqId] = { group, imgs, markers: pointMarkers };
        });
        
        const dropdown = document.getElementById('seq-dropdown');
        SEQUENCES.forEach(seqId => {
            const opt = document.createElement('option');
            opt.value = seqId;
            opt.textContent = `${seqId.substring(0, 12)}... (${(bySeq[seqId] || []).length} images)`;
            dropdown.appendChild(opt);
        });
        
        let currentSeq = SEQUENCES[0];
        let currentIdx = 0;
        let currentMarkers = [];
        let currentImgs = [];
        let showOverlay = false;
        
        document.getElementById('tab-orig').onclick = () => {
            showOverlay = false;
            document.getElementById('tab-orig').classList.add('active');
            document.getElementById('tab-overlay').classList.remove('active');
            updatePhoto();
        };
        document.getElementById('tab-overlay').onclick = () => {
            showOverlay = true;
            document.getElementById('tab-overlay').classList.add('active');
            document.getElementById('tab-orig').classList.remove('active');
            updatePhoto();
        };
        
        function switchSequence(seqId) {
            currentSeq = seqId;
            currentIdx = 0;
            
            Object.entries(seqLayers).forEach(([sid, layer]) => {
                if (sid === seqId) map.addLayer(layer.group);
                else map.removeLayer(layer.group);
            });
            
            currentMarkers = seqLayers[seqId].markers;
            currentImgs = seqLayers[seqId].imgs;
            
            goTo(0);
            
            if (currentImgs.length > 0) {
                const lats = currentImgs.map(d => d.lat);
                const lons = currentImgs.map(d => d.lon);
                map.fitBounds([
                    [Math.min(...lats), Math.min(...lons)],
                    [Math.max(...lats), Math.max(...lons)]
                ], { padding: [30, 30] });
            }
        }
        
        dropdown.onchange = (e) => switchSequence(e.target.value);
        
        function updatePhoto() {
            const d = currentImgs[currentIdx];
            if (!d) return;
            
            if (showOverlay && d.overlay_url) {
                document.getElementById('current-photo').src = d.overlay_url;
            } else {
                document.getElementById('current-photo').src = d.url;
            }
            updateStats(d);
        }
        
        function updateStats(d) {
            const el = document.getElementById('seg-stats');
            if (!d.segmentation) {
                el.innerHTML = '<span style="color:#666">No segmentation data</span>';
                return;
            }
            
            let html = '';
            Object.entries(d.segmentation).forEach(([group, pct]) => {
                if (pct < 0.5) return;
                const color = SEG_COLORS[group] || '#888';
                html += `<div class="seg-item">
                    <div class="seg-color" style="background:${color}"></div>
                    <span class="seg-label">${group}:</span>
                    <span class="seg-value">${pct}%</span>
                </div>`;
            });
            el.innerHTML = html;
        }
        
        // ═══════════════════════════════════════════
        // Plotly Chart
        // ═══════════════════════════════════════════
        function updatePlot(idx) {
            const upto = currentImgs.slice(0, idx + 1);
            const groups = ['building', 'sky', 'vegetation', 'road', 'sidewalk'];
            
            // ── Window: show last N points ──
            const WINDOW = 20;
            const start = Math.max(0.5, idx + 1 - WINDOW + 0.5);
            const end = idx + 0.5 + 0.5;
            
            // ── Build traces ──
            const traces = groups.map(g => ({
                x: upto.map((_, i) => i + 1),
                y: upto.map(d => 
                    (d.segmentation && d.segmentation[g] !== undefined) 
                        ? d.segmentation[g] 
                        : 0
                ),
                type: 'scatter',
                mode: 'lines+markers',
                name: g,
                line: { color: SEG_COLORS[g] || '#888', width: 2 },
                marker: { 
                    size: 5, 
                    color: SEG_COLORS[g] || '#888',
                    line: { color: '#fff', width: 1 },
                },
                hovertemplate: `<b>${g}</b>: %{y:.1f}%<extra></extra>`,
            }));
            
            // ── Layout ──
            const layout = {
                margin: { l: 40, r: 10, t: 20, b: 40 },
                
                xaxis: {
                    title: { text: 'Image #', font: { size: 10, color: '#aaa' } },
                    tickfont: { size: 9, color: '#aaa' },
                    gridcolor: 'rgba(255,255,255,0.1)',
                    
                    // Auto-follow: show last WINDOW points
                    range: [start, end],
                    
                    // Horizontal scrollbar
                    rangeslider: {
                        visible: true,
                        thickness: 0.08,
                        bgcolor: '#333',
                        bordercolor: '#555',
                        borderwidth: 1,
                    },
                    
                    // Allow pan/zoom
                    fixedrange: false,
                },
                
                yaxis: {
                    title: { text: '%', font: { size: 10, color: '#aaa' } },
                    tickfont: { size: 9, color: '#aaa' },
                    range: [0, 100],
                    gridcolor: 'rgba(255,255,255,0.1)',
                    fixedrange: true,
                },
                
                legend: {
                    orientation: 'h',
                    x: 0.5,
                    xanchor: 'center',
                    y: 1.15,
                    font: { size: 10, color: '#fff' },
                },
                
                paper_bgcolor: '#1a1a1a',
                plot_bgcolor: '#1a1a1a',
                showlegend: true,
                height: 200,
                dragmode: 'pan',
            };
            
            // ── Render ──
            Plotly.react('chart-container', traces, layout, {
                displayModeBar: false,
                responsive: true,
                scrollZoom: true,
            });
        }
        
        function goTo(idx) {
            if (idx < 0 || idx >= currentImgs.length) return;
            
            currentIdx = idx;
            const d = currentImgs[idx];
            
            updatePhoto();
            updatePlot(idx);
            map.panTo([d.lat, d.lon]);
            
            currentMarkers.forEach((m, i) => {
                if (!m.setStyle) return;
                m.setStyle({
                    radius: i === idx ? 10 : 5,
                    color: i === idx ? '#ff3333' : seqColors[currentSeq],
                    fillColor: i === idx ? '#ff3333' : seqColors[currentSeq],
                });
            });
            
            document.getElementById('counter').textContent = 
                `${idx + 1} / ${currentImgs.length}`;
            
            const ts = d.captured_at 
                ? new Date(d.captured_at).toLocaleString('en-US')
                : '';
            const coord = `${d.lat.toFixed(5)}, ${d.lon.toFixed(5)}`;
            const pano = d.is_pano ? ' 📷 Pano' : '';
            document.getElementById('info').innerHTML = 
                `<b>${coord}</b><br>${ts}${pano}`;
            
            document.getElementById('prev').disabled = (idx === 0);
            document.getElementById('next').disabled = (idx === currentImgs.length - 1);
        }
        
        document.getElementById('prev').onclick = () => goTo(currentIdx - 1);
        document.getElementById('next').onclick = () => goTo(currentIdx + 1);
        
        document.addEventListener('keydown', (e) => {
            if (e.key === 'ArrowLeft') goTo(currentIdx - 1);
            if (e.key === 'ArrowRight') goTo(currentIdx + 1);
        });
        
        switchSequence(SEQUENCES[0]);
    </script>
</body>
</html>"""

# ── Inject data ──
html = html.replace("META_JSON", json.dumps(meta))
html = html.replace("SEQ_JSON", json.dumps(sequences_list))

# ── Save ──
INDEX_HTML = PROJECT_ROOT / "index.html"
with open(INDEX_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print(f"\n✅ Saved: {INDEX_HTML}")
print(f"   {len(sequences_list)} sequences")
print(f"   {len(meta)} images")


✅ Saved: D:\DOWNLOADS\nasim\jupyters\maps\map\new code for git\Project_Mapilary_GIT\index.html
   2 sequences
   60 images


In [14]:
import webbrowser
webbrowser.open(f"file://{INDEX_HTML.absolute()}")

True